## Imports

In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.neighbors import KNeighborsRegressor

## Loading data

In [1]:
train = pd.read_csv('path/to/train.csv')
test = pd.read_csv('path/to/test.csv')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


NameError: name 'pd' is not defined

## EDA

#### General information

In [5]:
print("="*60)
print("1. GENERAL INFORMATION")
print("="*60)

# Data types
print("\nData types:")
print(train.dtypes.value_counts())

# Missing values
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nMissing values:")
print(f"  Total: {missing.sum()} ({missing.sum()/(train.shape[0]*train.shape[1])*100:.2f}%)")
print(f"  Columns with missing data: {len(missing)}")
if len(missing) > 0:
    print("\n  Top 5:")
    for col, val in missing.head(5).items():
        print(f"    {col}: {val} ({val/len(train)*100:.1f}%)")

# Target distribution
print(f"\nTarget (averageRating): mean={train['averageRating'].mean():.2f}, median={train['averageRating'].median():.2f}, std={train['averageRating'].std():.2f}")

plt.figure(figsize=(6, 4))
plt.boxplot(train['averageRating'], vert=True, patch_artist=True, boxprops=dict(facecolor='steelblue'))
plt.title('Average Rating Distribution')
plt.ylabel('Rating')
plt.grid(True, alpha=0.3)
plt.show()

=== Типы данных ===



NameError: name 'train' is not defined

#### Categorial features

In [ ]:
print("\n" + "="*60)
print("2. CATEGORICAL FEATURES ANALYSIS")
print("="*60)

categorical_columns = train.select_dtypes(include=['object']).columns

summary_data = []
for col in categorical_columns:
    n_unique = train[col].nunique()
    n_missing = train[col].isnull().sum()
    missing_pct = n_missing / len(train) * 100
    most_frequent = train[col].mode().iloc[0] if n_unique > 0 else 'None'
    top_freq_pct = train[col].value_counts().iloc[0] / len(train) * 100 if n_unique > 0 else 0
    
    summary_data.append({
        'Column': col,
        'Type': 'object',
        'Unique': n_unique,
        'Missing': f"{n_missing} ({missing_pct:.1f}%)",
        'Most Frequent': most_frequent,
        'Top Freq %': f"{top_freq_pct:.1f}%"
    })

summary_df = pd.DataFrame(summary_data)
print("\nCategorical features summary:")
print(summary_df.to_string(index=False))

if 'titleType' in categorical_columns:
    print("\n" + "-"*40)
    print("titleType value counts:")
    print(train['titleType'].value_counts().head(10))

#### Correlation analysis

In [ ]:
print("\n" + "="*60)
print("3. CORRELATION ANALYSIS")
print("="*60)

numeric_cols = train.select_dtypes(include=[np.number]).columns
corr_matrix = train[numeric_cols].corr()
target_corr = corr_matrix['averageRating'].drop('averageRating').sort_values(ascending=False)

print("\nCorrelation with averageRating:")
print("Top 5 positive:")
for col, val in target_corr.head(5).items():
    print(f"  {col}: {val:.3f}")
print("\nTop 5 negative:")
for col, val in target_corr.tail(5).items():
    print(f"  {col}: {val:.3f}")

plt.figure(figsize=(10, 8))
top_features = target_corr.abs().sort_values(ascending=False).head(10).index
corr_subset = train[list(top_features) + ['averageRating']].corr()
sns.heatmap(corr_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix - Top 10 Features + Target')
plt.tight_layout()
plt.show()

## Featute Engineering

In [ ]:
class FeatureAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in ["startYear", "endYear", "runtimeMinutes", "numVotes",
                    "mainDirectorBirthYear", "mainDirectorDeathYear",
                    "mainWriterBirthYear", "mainWriterDeathYear",
                    "titleLength", "genderVoters", "Downloads", "Cash"]:
            X[col] = pd.to_numeric(X[col], errors="coerce")
        X["log_numVotes"] = np.log1p(X["numVotes"].fillna(0).clip(lower=0))
        X["log_runtimeMinutes"] = np.log1p(X["runtimeMinutes"].fillna(0).clip(lower=0))
        X["log_genderVoters"] = np.log1p(X["genderVoters"].fillna(0).clip(lower=0))
        X["directorAgeAtRelease"] = X["startYear"] - X["mainDirectorBirthYear"]
        X["writerAgeAtRelease"] = X["startYear"] - X["mainWriterBirthYear"]
        X["directorDead"] = X["mainDirectorDeathYear"].notna().astype(int)
        X["writerDead"] = X["mainWriterDeathYear"].notna().astype(int)
        X["hasEndYear"] = X["endYear"].notna().astype(int)

        def _count_genres(g):
            try:
                parsed = ast.literal_eval(g)
                return sum(1 for x in parsed if x is not None)
            except:
                return 0

        X["genreCount"] = X["genres"].fillna("[]").apply(_count_genres)
        X["titleHasDigit"] = X["primaryTitle"].fillna("").apply(lambda t: int(any(c.isdigit() for c in t)))
        X["titleWordCount"] = X["primaryTitle"].fillna("").apply(lambda t: len(t.split()))
        X["titleIsOriginal"] = (X["primaryTitle"] == X["originalTitle"]).astype(int)
        X["isAdult"] = pd.to_numeric(X["isAdult"], errors="coerce").fillna(0).astype(int)
        votes = X["numVotes"].fillna(0).replace(0, 1)
        X["genderVoterRatio"] = X["genderVoters"].fillna(0) / votes
        return X

In [ ]:
class GenreEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, max_genres=30):
        self.max_genres = max_genres

    @staticmethod
    def _parse(g):
        try:
            parsed = ast.literal_eval(g)
            return [x for x in parsed if x is not None]
        except:
            return []

    def fit(self, X, y=None):
        all_genres = set()
        for val in X.iloc[:, 0].fillna("[]"):
            all_genres.update(self._parse(val))
        self.genres_ = sorted(all_genres)[:self.max_genres]
        return self

In [ ]:
class ProfessionEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, max_professions=15):
        self.max_professions = max_professions

    def fit(self, X, y=None):
        all_profs = set()
        for col_idx in range(X.shape[1]):
            for val in X.iloc[:, col_idx].fillna(""):
                for p in str(val).split(","):
                    p = p.strip()
                    if p:
                        all_profs.add(p)
        self.professions_ = sorted(all_profs)[:self.max_professions]
        return self

    def transform(self, X):
        result = np.zeros((len(X), len(self.professions_) * X.shape[1]), dtype=float)
        for col_idx in range(X.shape[1]):
            vals = X.iloc[:, col_idx].fillna("")
            for j, prof in enumerate(self.professions_):
                idx = col_idx * len(self.professions_) + j
                result[:, idx] = vals.apply(
                    lambda p: float(prof in [x.strip() for x in str(p).split(",")])).values
        return result

In [ ]:
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=20, min_count=3):
        self.smoothing = smoothing
        self.min_count = min_count

    def fit(self, X, y=None):
        self.maps_ = {}
        self.global_mean_ = None
        if y is not None:
            self.global_mean_ = y.mean()
            for col_idx in range(X.shape[1]):
                col_vals = X.iloc[:, col_idx]
                stats = pd.DataFrame({"col": col_vals, "target": y})
                agg = stats.groupby("col")["target"].agg(["mean", "count"])
                agg = agg[agg["count"] >= self.min_count]
                smooth = (agg["mean"] * agg["count"] + self.global_mean_ * self.smoothing) / (
                        agg["count"] + self.smoothing)
                self.maps_[col_idx] = smooth.to_dict()
        return self

    def transform(self, X):
        result = np.zeros((len(X), len(self.maps_)), dtype=float)
        for col_idx, tmap in self.maps_.items():
            result[:, col_idx] = X.iloc[:, col_idx].map(tmap).fillna(self.global_mean_).values
        return result

## Model

In [ ]:
class EnsembleRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, kr_alpha=0.3, kr_gamma=0.005,
                 ridge_alpha=5.0, knn_ks=(10, 20, 30, 50),
                 w_kr=0.65, w_ridge=0.05, w_knn=0.30):
        self.kr_alpha = kr_alpha
        self.kr_gamma = kr_gamma
        self.ridge_alpha = ridge_alpha
        self.knn_ks = knn_ks
        self.w_kr = w_kr
        self.w_ridge = w_ridge
        self.w_knn = w_knn

    def fit(self, X, y):
        self.kr_ = KernelRidge(alpha=self.kr_alpha, kernel="laplacian", gamma=self.kr_gamma)
        self.ridge_ = Ridge(alpha=self.ridge_alpha, random_state=42)
        self.knns_ = [KNeighborsRegressor(n_neighbors=k, weights="distance") for k in self.knn_ks]

        self.kr_.fit(X, y)
        self.ridge_.fit(X, y)
        for knn in self.knns_:
            knn.fit(X, y)
        return self

    def predict(self, X):
        pred_kr = self.kr_.predict(X)
        pred_ridge = self.ridge_.predict(X)
        pred_knn = np.mean([knn.predict(X) for knn in self.knns_], axis=0)
        predictions = self.w_kr * pred_kr + self.w_ridge * pred_ridge + self.w_knn * pred_knn
        return np.clip(predictions, 1.0, 10.0)

## Pipeline

In [ ]:
num_cols = [
    "startYear", "runtimeMinutes", "numVotes",
    "mainDirectorBirthYear", "mainWriterBirthYear",
    "titleLength", "genderVoters",
    "log_numVotes", "log_runtimeMinutes", "log_genderVoters",
    "directorAgeAtRelease", "writerAgeAtRelease",
    "directorDead", "writerDead", "hasEndYear",
    "genreCount", "genderVoterRatio",
    "titleHasDigit", "titleWordCount", "titleIsOriginal",
    "isAdult",
]

cat_cols = ["titleType"]
genre_cols = ["genres"]
prof_cols = ["MainDirectorPrimaryProfession", "MainWriterPrimaryProfession"]
target_enc_cols = ["titleType"]

pipe = Pipeline([
    ('feature_adder', FeatureAdder()),
    ('preprocessor', ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), num_cols),
            ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
            ('genre', GenreEncoder(max_genres=30), genre_cols),
            ('prof', ProfessionEncoder(max_professions=15), prof_cols),
            ('te', TargetEncoder(smoothing=20, min_count=3), target_enc_cols),
        ]
    )),
    ('scaler', StandardScaler()),
    ('model', EnsembleRegressor(
        kr_alpha=0.3, kr_gamma=0.005,
        ridge_alpha=5.0,
        knn_ks=(10, 20, 30, 50),
        w_kr=0.65, w_ridge=0.05, w_knn=0.30,
    )),
])